### Gradio 1st Application
## Company Brouchure Display using Gradio

In [132]:
import os
from dotenv import load_dotenv
from openai import OpenAI

In [108]:
import gradio as gr

In [133]:
load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")
google_api_key = os.getenv("GEMINI_API_KEY")
if openai_api_key:
    print("open api key exist")
else:
    print("open api key not valid")

if google_api_key:
    print("google api key exist")
else:
    print("google api key not valid")

open api key exist
google api key exist


In [110]:
openai = OpenAI(api_key=openai_api_key)

gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)

In [111]:
system_message = "You are a helpful assistant"


def message_gpt(prompt):
    messages = [{"role":"system", "content": system_message}, {"role":"user", "content":prompt}]
    response = openai.chat.completions.create(model='gpt-4.1-mini',messages=messages)
    return response.choices[0].message.content

In [94]:
message_gpt("What is today's date")

"Today's date is June 7, 2024."

In [10]:
def shout(text):
    print(f"shout has been called with some input {text}")
    return text.upper()

In [11]:
shout("Hello")

shout has been called with some input Hello


'HELLO'

In [ ]:
gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


shout has been called with some input hello
shout has been called with some input Hi tghere


In [16]:
gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode='never').launch(inbrowser=True, auth=('user','kanha'))

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


In [22]:
message_input= gr.Textbox(label="Your message :", info="Enter the message to be shouted", lines=7)
message_output = gr.Textbox(label="Response", lines=8)

view = gr.Interface(
    fn= message_gpt,
    title="GPT",
    inputs= [message_input],
    outputs= [message_output],
    examples = ["Hello","Amrita"],
    flagging_mode="never"
)
view.launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


In [30]:
system_message = "You are a helpful assistant that respond in Markdown without code blocks"

message_input = gr.Textbox(label="Your message :", info="Enter a message for gpt-4.1-mini", lines =7)
message_output = gr.Textbox(label="Response:")
view = gr.Interface(
    fn=message_gpt,
    title="GPT",
    inputs=[message_input],
    outputs=[message_output],
    examples=["Write a poem on AI", "What is Large Language Model ?"],
    flagging_mode="never"
)
view.launch()


* Running on local URL:  http://127.0.0.1:7869
* To create a public link, set `share=True` in `launch()`.


In [112]:
def stream_gpt(prompt):
    messages = [{"role":"system", "content": system_message}, {"role":"user", "content":prompt}]
    response = openai.chat.completions.create(
        model='gpt-4.1-mini',
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in response:
        result += chunk.choices[0].delta.content or ""
        yield result


In [96]:
system_message = "You are a helpful assistant that respond in Markdown without code blocks"

message_input = gr.Textbox(label="Your message :", info="Enter a message for gpt-4.1-mini", lines =7)
message_output = gr.Markdown(label="Response:")
view = gr.Interface(
    fn=stream_gpt,
    title="GPT",
    inputs=[message_input],
    outputs=[message_output],
    examples=["Write a poem on AI", "What is Large Language Model ?"],
    flagging_mode="never"
)
view.launch()

* Running on local URL:  http://127.0.0.1:7888
* To create a public link, set `share=True` in `launch()`.


In [118]:

def stream_gemini(prompt):
    messages = [{"role":"system", "content": system_message}, {"role":"user", "content":prompt}]
    response = openai.chat.completions.create(
        model='gemini-2.0-flash',
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in response:
        result += chunk.choices[0].delta.content or ""
        yield result

In [119]:
def stream_model(prompt, model):
    if model=="GPT":
        result = stream_gpt(prompt)
    elif model=="Gemini":
        result = stream_gemini(prompt)
    else:
        raise ValueError("Unknown model")
    yield from result

In [120]:
message_input = gr.Textbox(label="Your message:", info="Enter a message for the LLM", lines=7)
model_selector = gr.Dropdown(["GPT", "Gemini"], label="Select model", value="GPT")
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_model,
    title="LLMs", 
    inputs=[message_input, model_selector], 
    outputs=[message_output], 
    examples=[
            ["Explain the Transformer architecture to a layperson", "GPT"],
            ["Explain the Transformer architecture to an aspiring AI engineer", "Gemini"]
        ], 
    flagging_mode="never"
    )
view.launch()


* Running on local URL:  http://127.0.0.1:7896
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "/Users/am20322169/Desktop/LLM_Engineering/.venv/lib/python3.12/site-packages/gradio/queueing.py", line 766, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/am20322169/Desktop/LLM_Engineering/.venv/lib/python3.12/site-packages/gradio/route_utils.py", line 355, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/am20322169/Desktop/LLM_Engineering/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 2157, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/am20322169/Desktop/LLM_Engineering/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 1646, in call_function
    prediction = await utils.async_iteration(iterator)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/am20322169/Desktop/LLM_Engi

# Building a Company brouchure Generator

In [71]:
from scraper import fetch_website_contents

In [72]:
system_message = """
You are an assistant that analyzes the contents of a company website landing page
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
"""

In [99]:
def stream_brouchure(company_name,url,model):
    yield ""
    prompt = f"Please generate a company brochure for {company_name}. Here is their landing page:\n"
    prompt += fetch_website_contents(url)
    if model=='GPT':
        result = stream_gpt(prompt)
    elif model=='Gemini':
        result = stream_gemini(prompt)
    else:
        raise ValueError("Unknown model")
    yield from result

In [116]:
name_input = gr.Textbox(label='Company name:')
url_input = gr.Textbox(label="Landing page URL including http:// or https://")
model_selector = gr.Dropdown(["GPT","Gemini"], label="select a model", value="GPT")
message_output = gr.Markdown(label="Response:")
view = gr.Interface(
    fn = stream_brouchure,
    title="Company Brouchure Generator",
    inputs = [name_input, url_input, model_selector],
    outputs = [message_output],
    examples=[
            ["Hugging Face", "https://huggingface.co", "GPT"],
            ["Edward Donner", "https://edwarddonner.com", "Gemini"]
        ], 
    flagging_mode="never"
)
view.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7894
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "/Users/am20322169/Desktop/LLM_Engineering/.venv/lib/python3.12/site-packages/gradio/queueing.py", line 856, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/am20322169/Desktop/LLM_Engineering/.venv/lib/python3.12/site-packages/gradio/route_utils.py", line 355, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/am20322169/Desktop/LLM_Engineering/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 2157, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/am20322169/Desktop/LLM_Engineering/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 1646, in call_function
    prediction = await utils.async_iteration(iterator)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/am20322169/Desktop/LLM_Engi

In [141]:

deepseek_api_key = os.getenv("DEEPSEEK_API_KEY")
deepseek_url = "https://api.deepseek.com/v1"
deep_seek_api_key = OpenAI(api_key=deepseek_api_key, base_url=deepseek_url)

In [143]:
def stream_deepseek(prompt):
    messages = [{"role":"system", "content": system_message}, {"role":"user", "content":prompt}]
    response = openai.chat.completions.create(
        model='deepseek-coder',
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in response:
        result += chunk.choices[0].delta.content or ""
        yield result

In [144]:
def stream_brouchure(company_name,url,model):
    yield ""
    prompt = f"Please generate a company brochure for {company_name}. Here is their landing page:\n"
    prompt += fetch_website_contents(url)
    if model=='GPT':
        result = stream_gpt(prompt)
    elif model=='Gemini':
        result = stream_gemini(prompt)
    elif model=='Deepseek':
        result = stream_deepseek(prompt)
    else:
        raise ValueError("Unknown model")
    yield from result

In [ ]:
name_input = gr.Textbox(label='Company name:')
url_input = gr.Textbox(label="Landing page URL including http:// or https://")

model_selector = gr.Dropdown(["GPT","Gemini","Deepseek"], label="select a model", value="GPT")
message_output = gr.Markdown(label="Response:")
view = gr.Interface(
    fn = stream_brouchure,
    title="Company Brouchure Generator",
    inputs = [name_input, url_input, model_selector],
    outputs = [message_output],
    examples=[
            ["Hugging Face", "https://huggingface.co", "GPT"],
            ["Edward Donner", "https://edwarddonner.com", "Gemini"],
            ["Edward Donner", "https://edwarddonner.com", "Deepseek"]
        ], 
    flagging_mode="never"
)
view.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7902
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "/Users/am20322169/Desktop/LLM_Engineering/.venv/lib/python3.12/site-packages/gradio/queueing.py", line 856, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/am20322169/Desktop/LLM_Engineering/.venv/lib/python3.12/site-packages/gradio/route_utils.py", line 355, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/am20322169/Desktop/LLM_Engineering/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 2157, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/am20322169/Desktop/LLM_Engineering/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 1646, in call_function
    prediction = await utils.async_iteration(iterator)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/am20322169/Desktop/LLM_Engi

In [134]:
groq_api_key = os.getenv("GROQ_API_KEY")
groq_url = "https://api.groq.com/openai/v1"
groq_api_key = OpenAI(api_key=groq_api_key, base_url=groq_url)

In [138]:
def stream_groq(prompt):
    messages = [{"role":"system", "content": system_message}, {"role":"user", "content":prompt}]
    response = openai.chat.completions.create(
        model='llama3-70b-8192',
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in response:
        result += chunk.choices[0].delta.content or ""
        yield result

In [139]:
def stream_brouchure(company_name,url,model):
    yield ""
    prompt = f"Please generate a company brochure for {company_name}. Here is their landing page:\n"
    prompt += fetch_website_contents(url)
    if model=='GPT':
        result = stream_gpt(prompt)
    elif model=='Gemini':
        result = stream_gemini(prompt)
    elif model=='Deepseek':
        result = stream_deepseek(prompt)
    elif model=='Groq':
        result = stream_groq(prompt)  
    else:
        raise ValueError("Unknown model")
    yield from result

In [140]:
name_input = gr.Textbox(label='Company name:')
url_input = gr.Textbox(label="Landing page URL including http:// or https://")
model_selector = gr.Dropdown(["GPT","Gemini","Deepseek","Groq"], label="select a model", value="GPT")
message_output = gr.Markdown(label="Response:")
view = gr.Interface(
    fn = stream_brouchure,
    title="Company Brouchure Generator",
    inputs = [name_input, url_input, model_selector],
    outputs = [message_output],
    examples=[
            ["Hugging Face", "https://huggingface.co", "GPT"],
            ["Edward Donner", "https://edwarddonner.com", "Gemini"],
            ["Edward Donner", "https://edwarddonner.com", "Deepseek"],
            ["Edward Donner", "https://edwarddonner.com", "Groq"]
        ], 
    flagging_mode="never"
)
view.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7901
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "/Users/am20322169/Desktop/LLM_Engineering/.venv/lib/python3.12/site-packages/gradio/queueing.py", line 856, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/am20322169/Desktop/LLM_Engineering/.venv/lib/python3.12/site-packages/gradio/route_utils.py", line 355, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/am20322169/Desktop/LLM_Engineering/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 2157, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/am20322169/Desktop/LLM_Engineering/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 1646, in call_function
    prediction = await utils.async_iteration(iterator)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/am20322169/Desktop/LLM_Engi